In [ ]:
import kagglehub
import pandas as pd
import os
import ast
import html
import re

In [ ]:
# Download latest version of the original dataset with 48 features
path_threnjen = kagglehub.dataset_download("threnjen/board-games-database-from-boardgamegeek")

print("Path to dataset files:", path_threnjen)

# Download latest version of the simple dataset which contains mechanics
path_andrewmvd = kagglehub.dataset_download("andrewmvd/board-games")

print("Path to dataset files:", path_andrewmvd)

# Download latest version of the detailed dataset with several csv-files
path_jvanelteren = kagglehub.dataset_download("jvanelteren/boardgamegeek-reviews")

print("Path to dataset files:", path_jvanelteren)

In [ ]:
# The BGG descriptions contain HTML entities (&#10; for newlines, &quot;, &mdash; ...), some escaped
# twice (&amp;#039;), and UTF-8 text that was decoded as Windows-1252 (花火 shows up as è±ç«).
# Bytes 0x80-0x9F that Windows-1252 leaves undefined come through as raw control characters
_cp1252_high = bytes(range(0x80, 0xA0)).decode("cp1252", errors="ignore")
_cont = f"[\x80-\xbf{_cp1252_high}]"
_mojibake = re.compile(f"[\xc2-\xdf]{_cont}|[\xe0-\xef]{_cont}{{2}}|[\xf0-\xf4]{_cont}{{3}}")

def _fix_mojibake(match):
    """Turn a garbled character sequence back into its bytes and decode them as UTF-8."""
    raw = bytes(ord(ch) if ord(ch) < 256 else ch.encode("cp1252")[0] for ch in match.group())
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return match.group()

def clean_text(s):
    """Unescape HTML entities and repair mis-decoded UTF-8, repeating until the text stops changing."""
    if not isinstance(s, str):
        return s
    prev = None
    while prev != s:
        prev, s = s, _mojibake.sub(_fix_mojibake, html.unescape(s))
    # Drop leftover control characters that couldn't be repaired
    return re.sub("[\x80-\x9f]", "", s)

def clean_text_columns(df):
    """Apply clean_text to every text column of a dataframe."""
    for col in df.select_dtypes(["object", "string"]).columns:
        df[col] = df[col].map(clean_text)
    return df

In [ ]:
df_OG = clean_text_columns(pd.read_csv(os.path.join(path_threnjen, "games.csv")))
df_w_mechanics = clean_text_columns(pd.read_csv(os.path.join(path_andrewmvd, "bgg_dataset.csv"), sep=";"))
df_detailed = clean_text_columns(pd.read_csv(os.path.join(path_jvanelteren, "games_detailed_info2025.csv")))

In [ ]:
def is_missing(s):
    """True where a value is NaN, 0 (numeric columns) or an empty list/blank (text columns)."""
    if pd.api.types.is_numeric_dtype(s):
        return s.isna() | (s == 0)
    t = s.astype("string").str.strip()
    return s.isna() | t.isin(["", "[]", "0"])

In [ ]:
df_OG.head()

In [ ]:
df_w_mechanics.head()

In [ ]:
df_detailed.head()

In [ ]:
# Columns to keep from each dataframe
og_cols = [c for c in df_OG.columns
           if c not in ["Description", "StdDev", "ImagePath"] and not c.startswith("Rank:")]
detailed = df_detailed[["id", "description", "boardgamemechanic", "boardgamedesigner"]]
# ID is a float with a few NaNs, so drop those and cast to int to match BGGId
mechanics = df_w_mechanics[["ID", "Mechanics", "BGG Rank"]].dropna(subset=["ID"]).astype({"ID": int})

# Left join on df_OG so every game in the original dataset is kept
df_comb = (
    df_OG[og_cols]
    .merge(detailed, left_on="BGGId", right_on="id", how="left")
    .merge(mechanics, left_on="BGGId", right_on="ID", how="left")
    .drop(columns=["id", "ID"])
)

# boardgamemechanic and boardgamedesigner are lists stored as text, so turn them into real lists
for col in ["boardgamemechanic", "boardgamedesigner"]:
    df_comb[col] = df_comb[col].map(ast.literal_eval, na_action="ignore")
# Mechanics is comma-separated text. "Worker Placement, Different Worker Types" is one mechanic
# with a comma in its name, so don't split before "Different Worker Types"
df_comb["Mechanics"] = df_comb["Mechanics"].str.split(r", (?!Different Worker Types)", regex=True)

print(df_comb.shape)
df_comb.head()

In [ ]:
df_comb.iloc[0]["description"]

In [ ]:
# Calculate quantile boundaries based on AvgRating
quantiles = df_comb["AvgRating"].quantile([0.25, 0.50, 0.75])

# Take the 30 games whose AvgRating is closest to each quantile value
closest = [
    df_comb.loc[(df_comb["AvgRating"] - value).abs().nsmallest(30).index]
    .assign(Quantile=f"q{int(q * 100)}")
    for q, value in quantiles.items()
]
df_quantile_games = pd.concat(closest)[["BGGId", "Name", "AvgRating", "Quantile", "description"]]

# Save the 90 games to an Excel sheet instead of displaying them
excel_path = os.path.join("Data", "quantile_games.xlsx")
df_quantile_games.to_excel(excel_path, index=False)
print(f"Saved {len(df_quantile_games)} games to {excel_path}")

In [ ]:
# Save the combined dataframe as a csv in the Data folder
csv_path = os.path.join("Data", "combined_boardgames.csv")
df_comb.to_csv(csv_path, index=False)
print(f"Saved {df_comb.shape[0]} rows and {df_comb.shape[1]} columns to {csv_path}")